# Análise Exploratória e Decisões de Modelagem (dbt)

Este notebook consolida a análise inicial sobre os dados brutos armazenados no Google Cloud Storage, visando entender seus perfis, anomalias e justificar como essas descobertas embasaram a estrutura dos modelos Staging no projeto dbt.

Estaremos acessando diretamente os parquets no GCS usando DuckDB.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

### 1. Tabela Escola
Analisando a estrutura original em contraposição à documentação.

In [ ]:
con.execute("""
SELECT * FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/escola') LIMIT 5
""").df()

**Descoberta & Decisão:**
Notamos que as colunas documentadas `tipo` e `regiao` não estão presentes fisicamente no arquivo parquet (apenas `id_escola` e `bairro` existem). Para evitar que o pipeline falhe duramente e garantir que a estrutura da camada de apresentação contemple o contrato de dados esperado, o modelo `stg_educacao__escola` foi ajustado para projetar essas colunas faltantes como nulas (`cast(null as varchar) as tipo`, etc).

### 2. Tabela Avaliação: Desempenho por Faixa Etária vs. Disciplina
A tabela de avaliações apresenta um formato "wide" (disciplina_1 a disciplina_4) em vez do esperado "long".

In [ ]:
con.execute("""
WITH aval AS (
    SELECT id_aluno, AVG((disciplina_1 + disciplina_2 + disciplina_3 + disciplina_4) / 4.0) as media_geral
    FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/avaliacao')
    GROUP BY id_aluno
),
alunos AS (
    SELECT id_aluno, faixa_etaria
    FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/aluno')
)
SELECT faixa_etaria, ROUND(AVG(media_geral), 2) as media_faixa, COUNT(*) as qtd_alunos
FROM aval JOIN alunos USING (id_aluno)
GROUP BY 1 ORDER BY 1;
""").df()

**Descoberta & Decisão:**
As notas estão vindo como nulas em muitos registros agregados por aluno (ou seja, `media_geral` resulta em NaN se ao menos uma for nula). No entanto, observamos que o volume majoritário de alunos está na faixa `15-17` anos. O formato wide das disciplinas forçou o dbt staging (`stg_educacao__avaliacao`) a projetar as colunas individualmente em vez de depender de uma coluna única de nota. Os testes do dbt_expectations garantem que, quando a nota existe, está entre 0 e 10.

### 3. Tabela Frequência: Detecção de Anomalias
Analisando a distribuição das taxas médias de frequência.

In [ ]:
con.execute("""
SELECT ROUND(frequencia, 1) as bucket_freq, COUNT(*) as qtd_registros
FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/frequencia')
GROUP BY 1 
ORDER BY 2 DESC 
LIMIT 10;
""").df()

**Descoberta & Decisão:**
A distribuição indica uma hiper-concentração de frequências gravadas massivamente como `100.0` (mais de 1.8 milhões de registros exatos). Isso levanta um alerta de Qualidade de Dados (preenchimento "mecânico/default" por parte das escolas). Foi por isso que o teste customizado de regra de negócio foi desenvolvido para validar que não temos limites inválidos.

### 4. Disparidade de Avaliações por Bimestre (Missing Data)
Verificando a completude dos dados avaliativos ao longo do ano.

In [ ]:
con.execute("""
SELECT bimestre, COUNT(DISTINCT id_aluno) as alunos_avaliados
FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/avaliacao')
GROUP BY 1 ORDER BY 1;
""").df()

**Descoberta & Decisão:**
Há uma queda gradativa na quantidade de alunos avaliados do `bimestre 1` (66k) até o `bimestre 4` (44k). Isso evidencia clara Evasão Escolar (drop-out) ou Falha de Lançamento de Notas no último bimestre. Testes de `not_null` no dbt não quebram pois a linha sequer existe, mas o analista de BI precisará ser avisado sobre isso ao ler as documentações dos marts.

### 5. Concentração Demográfica (Alunos vs Escolas)
Para análises futuras sobre transporte e mobilidade escolar.

In [ ]:
con.execute("""
WITH alunos AS (
    SELECT id_aluno, bairro as bairro_aluno, id_turma
    FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/aluno')
),
frequencia AS (
    SELECT DISTINCT id_aluno, id_escola FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/frequencia')
),
escolas AS (
    SELECT id_escola, bairro as bairro_escola
    FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/escola')
)
SELECT
    CASE WHEN a.bairro_aluno = e.bairro_escola THEN 'Mesmo Bairro' ELSE 'Bairro Diferente' END as deslocamento,
    COUNT(DISTINCT a.id_aluno) as qtd_alunos
FROM alunos a
JOIN frequencia f ON a.id_aluno = f.id_aluno
JOIN escolas e ON f.id_escola = e.id_escola
GROUP BY 1;
""").df()

**Descoberta & Decisão:**
Nossos dados mostram que a vasta maioria/totalidade dos alunos estudam em escolas localizadas em "Bairro Diferente" do seu cadastro residencial, destacando uma possível política de roteamento da rede pública ou um desalinhamento dos IDs mascarados. Essa métrica é essencial para um Data Mart focado na área da demografia educacional.